In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f'Số ảnh train: {len(train_dataset)},  test: {len(test_dataset)}')

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.85MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 154kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.68MB/s]

Số ảnh train: 60000,  test: 10000


In [3]:
class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)   # 28→26
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)  # 13→11
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1   = nn.Linear(32 * 5 * 5, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # → (16, 13, 13)
        x = self.pool(torch.relu(self.conv2(x)))   # → (32,  5,  5)
        x = x.view(x.size(0), -1)                  # flatten
        x = self.fc1(x)                            # logits
        return x

model = MNIST_CNN().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'Tổng tham số: {n_params:,}')

MNIST_CNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=800, out_features=10, bias=True)
)
Tổng tham số: 12,810


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

In [ ]:
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss_sum += criterion(outputs, labels).item() * images.size(0)
            correct  += (outputs.argmax(1) == labels).sum().item()
            total    += labels.size(0)
    return loss_sum / total, correct / total

num_epochs = 5
loss_history, acc_history = [], []
test_loss_history, test_acc_history = [], []

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total
    test_loss, test_acc = evaluate(model, test_loader)
    loss_history.append(train_loss); acc_history.append(train_acc)
    test_loss_history.append(test_loss); test_acc_history.append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs}  '
          f'train_loss={train_loss:.4f}  train_acc={train_acc*100:.2f}%  '
          f'test_loss={test_loss:.4f}  test_acc={test_acc*100:.2f}%')

Epoch 1/5  train_loss=0.1823  train_acc=94.30%  test_loss=0.0630  test_acc=97.95%
Epoch 2/5  train_loss=0.0650  train_acc=98.06%  test_loss=0.0567  test_acc=98.15%


In [ ]:
epochs = range(1, num_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, loss_history,      'o-', label='Train')
axes[0].plot(epochs, test_loss_history, 's-', label='Test')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Loss')
axes[1].plot(epochs, [a*100 for a in acc_history],      'o-', label='Train')
axes[1].plot(epochs, [a*100 for a in test_acc_history], 's-', label='Test')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('Accuracy')
plt.tight_layout(); plt.show()

In [ ]:
test_loss, test_acc = evaluate(model, test_loader)
print(f'Final test accuracy: {test_acc*100:.2f}%')
print(f'(MNIST-ANN ở bài trước đạt ~97-98%; CNN nhỏ này thường đạt 98.5-99%.)')

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    preds = model(images).argmax(1)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    img = images[i].cpu().squeeze() * 0.3081 + 0.1307   # un-normalize để nhìn cho đẹp
    axes[i].imshow(img, cmap='gray')
    color = 'green' if preds[i] == labels[i] else 'red'
    axes[i].set_title(f'Đoán: {preds[i].item()}\nThật: {labels[i].item()}', color=color)
    axes[i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
model.eval()
images, _ = next(iter(test_loader))
img = images[0].unsqueeze(0).to(device)

with torch.no_grad():
    fmap1 = torch.relu(model.conv1(img))           # (1, 16, 26, 26)
    fmap2 = torch.relu(model.conv2(model.pool(fmap1)))  # (1, 32, 11, 11)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes[0, 0].imshow(img.cpu().squeeze() * 0.3081 + 0.1307, cmap='gray')
axes[0, 0].set_title('Ảnh gốc'); axes[0, 0].axis('off')
for i in range(7):
    axes[0, i+1].imshow(fmap1[0, i].cpu(), cmap='gray')
    axes[0, i+1].set_title(f'conv1 #{i}'); axes[0, i+1].axis('off')
for i in range(8):
    axes[1, i].imshow(fmap2[0, i].cpu(), cmap='gray')
    axes[1, i].set_title(f'conv2 #{i}'); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

print('Quan sát: filter conv1 thường nhận các cạnh hoặc đường nét đơn giản.')
print('         filter conv2 nhận các đặc trưng phức tạp hơn (mảng, hình.)')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Setup
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}\n')

In [ ]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f'Số ảnh train: {len(train_dataset)}, test: {len(test_dataset)}\n')

criterion = nn.CrossEntropyLoss()

def evaluate(model, loader, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss_sum += criterion(outputs, labels).item() * images.size(0)
            correct  += (outputs.argmax(1) == labels).sum().item()
            total    += labels.size(0)
    return loss_sum / total, correct / total

In [ ]:
class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1   = nn.Linear(32 * 5 * 5, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

In [ ]:
print("=" * 70)
print("CÂU 1: TRAIN 10 EPOCHS")
print("=" * 70 + "\n")

model1 = MNIST_CNN().to(device)
optimizer1 = optim.SGD(model1.parameters(), lr=0.01, momentum=0.9)

num_epochs = 10
loss_history, acc_history = [], []
test_loss_history, test_acc_history = [], []

for epoch in range(num_epochs):
    model1.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer1.zero_grad()
        outputs = model1(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer1.step()
        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total
    test_loss, test_acc = evaluate(model1, test_loader, device)
    loss_history.append(train_loss)
    acc_history.append(train_acc)
    test_loss_history.append(test_loss)
    test_acc_history.append(test_acc)

    gap = (train_acc - test_acc) * 100
    print(f'Epoch {epoch+1:2d}/10  train_loss={train_loss:.4f}  train_acc={train_acc*100:.2f}%  '
          f'test_loss={test_loss:.4f}  test_acc={test_acc*100:.2f}%  gap={gap:.2f}%')

print("\n📊 PHÂN TÍCH CÂU 1:")
acc_ep5 = test_acc_history[4]
acc_ep10 = test_acc_history[9]
gap_ep5 = (acc_history[4] - test_acc_history[4]) * 100
gap_ep10 = (acc_history[9] - test_acc_history[9]) * 100

print(f"Test accuracy tại epoch 5: {acc_ep5*100:.2f}%")
print(f"Test accuracy tại epoch 10: {acc_ep10*100:.2f}%")
print(f"Cải thiện: {(acc_ep10 - acc_ep5)*100:.2f}%")
print(f"\nGap (train_acc - test_acc) tại epoch 5: {gap_ep5:.2f}%")
print(f"Gap (train_acc - test_acc) tại epoch 10: {gap_ep10:.2f}%")
print(f"\n Gap đã mở rộng → DẤU HIỆU OVERFITTING!")

In [ ]:
print("\n" + "=" * 70)
print("CÂU 2: THÊM TẦNG CONV3")
print("=" * 70 + "\n")

class MNIST_CNN_Deep(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1   = nn.Linear(64 * 2 * 2, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # → (16, 13, 13)
        x = self.pool(torch.relu(self.conv2(x)))   # → (32, 5, 5)
        x = torch.relu(self.conv3(x))              # → (64, 5, 5)
        x = self.pool(x)                           # → (64, 2, 2)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

model2 = MNIST_CNN_Deep().to(device)
n_params_1 = sum(p.numel() for p in model1.parameters() if p.requires_grad)
n_params_2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)

print(f"2-Conv model: {n_params_1:,} params")
print(f"3-Conv model: {n_params_2:,} params (+{n_params_2 - n_params_1:,})")
print(f"\nTính toán kích thước feature map Conv3:")
print(f"  Input: (32, 5, 5)")
print(f"  Conv3 (k=3, p=1): (64, 5, 5)")
print(f"  Pool 2×2: (64, 2, 2)")
print(f"  Flatten: 64 × 2 × 2 = 256")

optimizer2 = optim.SGD(model2.parameters(), lr=0.01, momentum=0.9)

test_acc_history_2 = []
print("\nTraining 3-Conv model...")
for epoch in range(num_epochs):
    model2.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer2.zero_grad()
        outputs = model2(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer2.step()
        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)

    test_loss, test_acc = evaluate(model2, test_loader, device)
    test_acc_history_2.append(test_acc)

    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:2d}/10: test_acc={test_acc*100:.2f}%')

print(f"\n📊 PHÂN TÍCH CÂU 2:")
print(f"2-Conv - Final test acc: {test_acc_history[9]*100:.2f}%")
print(f"3-Conv - Final test acc: {test_acc_history_2[9]*100:.2f}%")

In [ ]:
print("\n" + "=" * 70)
print("CÂU 3: THAY ĐỔI LEARNING RATE")
print("=" * 70 + "\n")

learning_rates = [0.001, 0.01, 0.1]
lr_results = {}

for lr in learning_rates:
    print(f'Training với lr = {lr}...')

    model_lr = MNIST_CNN().to(device)
    optimizer_lr = optim.SGD(model_lr.parameters(), lr=lr, momentum=0.9)

    train_losses = []
    test_accs = []

    for epoch in range(5):
        model_lr.train()
        running_loss, total = 0.0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer_lr.zero_grad()
            outputs = model_lr(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer_lr.step()
            running_loss += loss.item() * images.size(0)
            total += labels.size(0)

        train_loss = running_loss / total
        _, test_acc = evaluate(model_lr, test_loader, device)
        train_losses.append(train_loss)
        test_accs.append(test_acc)

    lr_results[lr] = {'train_losses': train_losses, 'test_accs': test_accs}
    print(f'  → Final test_acc: {test_accs[-1]*100:.2f}%\n')

print("📊 PHÂN TÍCH CÂU 3:")
for lr in learning_rates:
    losses = lr_results[lr]['train_losses']
    accs = lr_results[lr]['test_accs']
    loss_drop = losses[0] - losses[-1]

    print(f'lr = {lr}:')
    print(f'  Loss: {losses[0]:.4f} → {losses[-1]:.4f} (giảm {loss_drop:.4f})')
    print(f'  Test acc: {accs[-1]*100:.2f}%')
    if lr == 0.001:
        print(f'   Quá thấp: loss giảm rất chậm')
    elif lr == 0.01:
        print(f'   Vừa phải: loss giảm ổn định')
    else:
        print(f'    Quá cao: có thể bất ổn')
    print()


In [ ]:
print("=" * 70)
print("CÂU 4: TRỰC QUAN FEATURE MAPS")
print("=" * 70 + "\n")

model1.eval()
images, _ = next(iter(test_loader))
img = images[0].unsqueeze(0).to(device)

with torch.no_grad():
    fmap1 = torch.relu(model1.conv1(img))
    fmap1_pool = model1.pool(fmap1)
    fmap2 = torch.relu(model1.conv2(fmap1_pool))
    fmap2_pool = model1.pool(fmap2)

print(f'Kích thước feature maps:')
print(f'  Ảnh gốc: {img.shape}')
print(f'  Conv1: {fmap1.shape}')
print(f'  Pool1: {fmap1_pool.shape}')
print(f'  Conv2: {fmap2.shape}')
print(f'  Pool2: {fmap2_pool.shape}')

print(f"\n📊 SO SÁNH FEATURE MAPS:")
print(f"  Conv1: Phát hiện các đặc trưng CƠ BẢN (cạnh, đường ngang/dọc)")
print(f"  Conv2: Phát hiện các đặc trưng PHỨC TẠP (hình dạng, mảng)")
print(f"  → Hierarchical Learning: từ thấp đến cao, từ đơn giản đến phức tạp")

In [ ]:
print("\n" + "=" * 70)
print("CÂU 5: DROPOUT + DATA AUGMENTATION")
print("=" * 70 + "\n")

train_transform_aug = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset_aug = torchvision.datasets.MNIST(root='./data', train=True,
                                               download=True, transform=train_transform_aug)
test_dataset_aug  = torchvision.datasets.MNIST(root='./data', train=False,
                                               download=True, transform=test_transform)

train_loader_aug = torch.utils.data.DataLoader(train_dataset_aug, batch_size=64, shuffle=True)
test_loader_aug  = torch.utils.data.DataLoader(test_dataset_aug,  batch_size=64, shuffle=False)

print('✓ Data augmentation: RandomAffine(rotation=±10°, translation=±10%)')
print('✓ Chỉ áp dụng cho train set\n')

class MNIST_CNN_Dropout(nn.Module):
    def __init__(self, dropout_p=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=0)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc1   = nn.Linear(32 * 5 * 5, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

print('1. BASELINE (không Dropout, không Augmentation):')
model_baseline = MNIST_CNN().to(device)
optimizer_baseline = optim.SGD(model_baseline.parameters(), lr=0.01, momentum=0.9)

test_accs_baseline = []
for epoch in range(num_epochs):
    model_baseline.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_baseline.zero_grad()
        outputs = model_baseline(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_baseline.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    test_loss, test_acc = evaluate(model_baseline, test_loader, device)
    test_accs_baseline.append(test_acc)

    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1:2d}: test_acc={test_acc*100:.2f}%')

print('\n2. DROPOUT + AUGMENTATION:')
model_aug = MNIST_CNN_Dropout(dropout_p=0.25).to(device)
optimizer_aug = optim.SGD(model_aug.parameters(), lr=0.01, momentum=0.9)

test_accs_aug = []
for epoch in range(num_epochs):
    model_aug.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.to(device)
        optimizer_aug.zero_grad()
        outputs = model_aug(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_aug.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    test_loss, test_acc = evaluate(model_aug, test_loader_aug, device)
    test_accs_aug.append(test_acc)

    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1:2d}: test_acc={test_acc*100:.2f}%')

print(f'\n📊 PHÂN TÍCH CÂU 5:')
print(f'Baseline - Epoch 5: {test_accs_baseline[4]*100:.2f}%, Epoch 10: {test_accs_baseline[9]*100:.2f}%')
print(f'Dropout+Aug - Epoch 5: {test_accs_aug[4]*100:.2f}%, Epoch 10: {test_accs_aug[9]*100:.2f}%')
print(f'\n✓ Dropout + Augmentation giúp model ổn định và khái quát hóa tốt hơn')